# Solving the XOR Problem Using a Neural Network in Keras

## 📚 Learning Objectives

By completing this notebook, you will:
- Solve the XOR problem using neural networks
- Use Keras to build neural networks
- Understand why single-layer networks fail
- Implement multi-layer solutions
- Train and evaluate the model

## 🔗 Where this fits

**Builds on:** Unit 3, lesson 02 "The Neuron, Perceptron, and XOR Problem" — the same XOR failure, now fixed with a hidden layer in Keras.

**Used later in:** Unit 3, lesson 04, which opens the black box behind the `loss:` Keras prints here, and Unit 4, which builds every later network on this `Sequential`/`Dense`/`fit` pattern.

---

This notebook covers practical activities from **Course 01, Unit 3**:
- Solving the XOR problem using a neural network in Keras

---

## Introduction

The **XOR problem** is a classic example demonstrating why multi-layer neural networks are necessary, as single-layer perceptrons cannot solve non-linearly separable problems.


## 🎯 The door out of the fifteen-year winter

The previous notebook showed the wall: Minsky and Papert proved in **1969** that a
single-layer perceptron cannot compute XOR, and interest in neural networks
collapsed. The door was published in **1986**, when David Rumelhart, Geoffrey
Hinton and Ronald Williams presented backpropagation in *Nature* — a practical way
to send the error signal backwards through a hidden layer so its weights could be
trained too. Every network in this diploma, and every one in industry, is trained
by the descendant of that algorithm.

Three years later Cybenko (1989), and then Hornik (1991), proved the other half:
a network with **one hidden layer** and a non-linear activation can approximate
any continuous function on a bounded region, given enough units. Together those
two results answer both halves of the 1969 objection — one layer of hidden units
is enough to *represent* the answer, and backpropagation is enough to *find* it.

The four rows below are the smallest possible demonstration of a fact that
changed the field.

### What goes wrong without a hidden layer

Nothing dramatic: the model trains, the loss goes down, and it stops at the best
straight line — which for XOR is 50% or 75% accuracy, depending on which line it
settles on. There is no error, no warning, no diagnostic. The model quietly
returns the best answer inside a hypothesis space that does not contain the right
one. Recognising that situation is the skill; the hidden layer is just the fix.


## 📥 Inputs & 📤 Outputs

**Inputs:** What we use in this notebook

- Libraries and concepts as introduced in this notebook; see prerequisites and code comments.

**Outputs:** What you'll see when you run the cells

- Printed results, figures, and summaries as shown when you run the cells.

---

In [1]:
# Setup: the XOR truth table as a 4-row dataset. XOR matters historically — it is the
# simplest function a single-layer perceptron provably cannot learn.
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("✅ Libraries imported!")
print("\nSolving XOR Problem with Keras")
print("=" * 60)

# XOR truth table
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y = np.array([[0], [1], [1], [0]])

print("\nXOR Truth Table:")
print("Input | Output")
print("------|-------")
for i in range(len(X)):
    print(f"{X[i]} | {y[i][0]}")

print("\n✅ XOR data prepared!")

✅ Libraries imported!

Solving XOR Problem with Keras

XOR Truth Table:
Input | Output
------|-------
[0 0] | 0
[0 1] | 1
[1 0] | 1
[1 1] | 0

✅ XOR data prepared!


In [2]:
# Build the smallest network that CAN learn XOR: one hidden ReLU layer bends the
# decision boundary; the sigmoid output turns the result into a probability.
# Build multi-layer neural network
print("=" * 60)
print("BUILDING NEURAL NETWORK")
print("=" * 60)

# fixed seed: without it, small ReLU nets sometimes fail to solve XOR (dead units)
keras.utils.set_random_seed(42)

model = keras.Sequential([
    layers.Dense(4, activation='relu', input_shape=(2,)),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam', loss='binary_crossentropy',
    metrics=['accuracy']
)

print("\nModel architecture:")
model.summary()

print("\n✅ Model built!")

BUILDING NEURAL NETWORK

Model architecture:


/Users/abdullah/venvs/ai-diploma-tf/lib/python3.13/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 4)              │            12 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │             5 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 17 (68.00 B)

 Trainable params: 17 (68.00 B)

 Non-trainable params: 0 (0.00 B)


✅ Model built!


In [3]:
# Train on all 4 rows for 1000 epochs — watch the loss: it should approach 0 as the
# network carves the plane into the XOR pattern.
# Train the model
print("=" * 60)
print("TRAINING MODEL")
print("=" * 60)

history = model.fit(
    X, y,
    epochs=1000,
    verbose=0,
    batch_size=4
)

print(f"\nTraining completed!")
print(f"Final loss: {history.history['loss'][-1]:.4f}")
print(f"Final accuracy: {history.history['accuracy'][-1]:.4f}")

print("\n✅ Model trained!")

TRAINING MODEL



Training completed!
Final loss: 0.1663
Final accuracy: 1.0000

✅ Model trained!


In [4]:
# Evaluate on the same 4 rows (the ENTIRE universe of XOR inputs): a correctly trained
# hidden layer scores 100%, which the single perceptron could never do.
import numpy as np
# Evaluate the model
print("=" * 60)
print("EVALUATING MODEL")
print("=" * 60)

predictions = model.predict(X, verbose=0)
predicted_classes = (predictions > 0.5).astype(int)

print("\nPredictions:")
print("Input | Expected | Predicted | Correct")
print("------|----------|-----------|--------")
for i in range(len(X)):
    correct = "✓" if predicted_classes[i][0] == y[i][0] else "✗"
    print(f"{X[i]} | {y[i][0]}        | {predicted_classes[i][0]}          | {correct}")

accuracy = np.mean(predicted_classes.flatten() == y.flatten())
print(f"\nAccuracy: {accuracy:.2%}")

print("\n✅ Model evaluated!")

EVALUATING MODEL

Predictions:
Input | Expected | Predicted | Correct
------|----------|-----------|--------
[0 0] | 0        | 0          | ✓
[0 1] | 1        | 1          | ✓
[1 0] | 1        | 1          | ✓
[1 1] | 0        | 0          | ✓

Accuracy: 100.00%

✅ Model evaluated!


## 📊 The comparison across notebooks 02 and 03

Same problem, same four rows, one architectural change:

| notebook | architecture | seed fixed? | accuracy on XOR |
|---|---|---|---|
| 02 | hidden layer, unseeded run | no | **50%** (2 of 4 — a coin flip) |
| **03 (this one)** | **`Dense(4, relu)` → `Dense(1, sigmoid)`** | **yes** | **100%** (4 of 4) |

**The conclusion these two runs support:** the hidden layer makes the correct
answer *representable*, and a good initialisation makes it *findable*. Both are
required. Notebook 02 had the capacity and lost the initialisation lottery;
this notebook fixed the seed and won it.

**One number deserves a second look.** Training ended with `Final loss: 0.1663`
and `Final accuracy: 1.0000`. Perfect accuracy, but a loss well above zero —
which means the sigmoid outputs are on the right side of 0.5 without being near 0
and 1. The network is *correct but unconfident*. Accuracy rounds that away;
the loss does not. This is why you read both.


## 💬 Discuss

1. We evaluated the model on the same four rows it trained on — normally a
   cardinal sin. Here it is defensible, because those four rows are the *entire*
   universe of XOR inputs; there is no fifth case to hold out. **State the general
   rule this exception reveals**, then name a real problem where you might
   genuinely be able to test on all possible inputs.
2. Accuracy said 1.0000 and loss said 0.1663. Which would you report to a
   manager, which to an engineer, and what would you do if a colleague showed you
   only the first one? Use the numbers to justify your answer.
3. The seed is fixed with a comment explaining that without it the network
   sometimes fails. Is fixing the seed good scientific practice, or is it
   presenting a lucky run as a result? **Argue both sides**, then say what you
   would actually put in a paper or a project report.


## Summary

This notebook covered:
- ✅ **XOR Problem**: Non-linearly separable problem requiring multi-layer networks
- ✅ **Neural Network**: Built with Keras using hidden layer
- ✅ **Training**: Successfully solved XOR with multi-layer architecture
- ✅ **Evaluation**: Achieved 100% accuracy on XOR truth table

The XOR problem demonstrates why multi-layer neural networks are essential for complex problems.

## ⚠️ Where this breaks

- **The reported 100% is one seed's result.** The code fixes `random_state`
  specifically because small ReLU networks on four points sometimes land with dead
  units and never recover — notebook 02 is that outcome, printed. A fixed seed
  makes the demo reproducible; it does **not** make the architecture reliable, and
  quoting a single seeded run as "the accuracy" is the most common way students
  overstate a neural-network result.
- **Train and test are the same four rows.** No generalisation is being measured
  at all. That is acceptable *only* because XOR's input space is exactly these
  four points. Repeat this pattern on any real dataset and the number you print is
  memorisation, not performance — which is precisely the failure Unit 4,
  notebook 07 is about.
- **1,000 epochs on 4 samples is not a training budget you can reuse.** With
  `batch_size=4` that is 1,000 optimizer steps spent on a four-row table.
  Nothing here tells you how long a real model needs.
- **Universal approximation is an existence result, not a recipe.** Cybenko's
  theorem says a suitable one-hidden-layer network *exists*; it says nothing about
  how many units it needs, whether gradient descent will find it, or how much data
  it would take. In practice depth is usually cheaper than width, which is why the
  networks in Unit 4 are deep rather than very wide.
- **When not to reach for a network at all:** if you know the interaction that
  makes your problem non-linear, encode it as a feature. Add `x1 * x2` to the XOR
  table and a single linear model solves it exactly, trains instantly, and can be
  read by a human. A hidden layer is what you use when you *cannot* name the
  interaction — and it costs you the ability to explain it.


## 📚 References

1. Rumelhart, D. E., Hinton, G. E., & Williams, R. J. (1986). *Learning Representations by Back-Propagating Errors*. Nature, 323, 533–536.
2. Goodfellow, I., Bengio, Y., & Courville, A. (2016). *Deep Learning*, Ch. 6: Deep Feedforward Networks (includes the XOR example). MIT Press.
3. Kingma, D. P., & Ba, J. (2015). *Adam: A Method for Stochastic Optimization*. ICLR. <https://arxiv.org/abs/1412.6980>